## MVP ANA - Semantic Frequency Analysis of Brazilian Legal Documents

### Project Overview

This notebook implements a full NLP pipeline that transforms raw PDF legal documents into structured semantic frequency statistics. Starting from scanned or digital court documents, the pipeline extracts sentences, normalises encoding artifacts, removes boilerplate, segments into sentences, normalises citations, and clusters semantically similar Súmulas using BERTopic (UMAP + HDBSCAN) with automatic three-level topic labels derived from STJ document headers and c-TF-IDF keywords.

### Domain Context: Brazilian Legal Document Analysis

#### Core Domain Definition
Brazilian court documents (petições, acórdãos, sentenças, despachos) address a wide variety of criminal case types. Clustering all sentences together and labelling clusters post-hoc via regex allows pattern discovery across the full corpus, revealing recurring arguments, standard precedents, and emergent legal themes that may not match any predefined case type.

### Problem Definition: Semantic Frequency Analysis Pipeline

#### Problem Statement
Develop a pipeline that ingests Brazilian legal PDFs and outputs frequency statistics of semantically grouped sentences per criminal case type. The statistics surface recurring thematic clusters, enabling researchers and practitioners to identify dominant legal arguments, frequently cited precedents, and standard procedural language at scale.

#### Technical Requirements
- **Problem Type**: Unsupervised clustering with post-hoc case type labeling
- **Input Format**: Brazilian legal PDFs (digital or scanned)
- **Output Format**: Cluster frequency tables, similarity matrices, and static visualizations
- **Difficulty Level**: High — multilingual NLP, OCR, domain-specific abbreviations, sparse legal corpora

#### Data Landscape
The pipeline processes:
- Raw PDF pages (digital text layer or scanned images)
- 1024-dim BERT embeddings per sentence (neuralmind/bert-large-portuguese-cased)
- UMAP 5D projection for HDBSCAN clustering + separate 2D projection for visualization
- HDBSCAN cluster assignments with outlier reassignment via reduce_outliers
- Three-level topic labels: Branch > Subject > c-TF-IDF_subtype (extracted from document headers + statistical keywords)

### References

- **Architecture Decision Record**: [semantic_analysis_solution_adr.docx](docs/semantic_analysis_solution_adr.docx) — Documents the pipeline design, technology choices, and rationale for each step;
- **BERTimbau Large model**: [neuralmind/bert-large-portuguese-cased](https://huggingface.co/neuralmind/bert-large-portuguese-cased) — Large Portuguese BERT used for sentence embeddings, providing richer 1024-dim representations than the base variant;

---

This notebook serves as the primary entry point for the MVP ANA implementation. Each pipeline step is self-contained and its output is checkpointed to disk, allowing individual steps to be re-run without re-executing expensive upstream operations.

### Import Libraries

Installation and import of all required libraries. Warning verbosity is suppressed globally to keep notebook output clean.

In [1]:
%pip install pdfplumber pytesseract Pillow ftfy scikit-learn umap-learn hdbscan transformers torch numpy matplotlib bertopic faiss-cpu

import warnings
import logging
import importlib
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add pipeline scripts to Python path
scripts_path = Path("./pipeline/scripts")
if str(scripts_path.resolve()) not in sys.path:
    sys.path.insert(0, str(scripts_path.resolve()))

# Import and reload all pipeline step modules to pick up changes without kernel restart
import pipeline_step
import pipeline_manager
import step_0_pdf_reader
import step_1_encoding_normalizer
import step_2_boilerplate_remover
import step_3_sentence_segmenter
import step_4_citation_normalizer
import step_5_embedding_generator
import step_6_bertopic
import step_7_statistics_generator
import step_8_visualizer
import step_9_xml_exporter
import step_10_search_index

importlib.reload(pipeline_step)
importlib.reload(pipeline_manager)
importlib.reload(step_0_pdf_reader)
importlib.reload(step_1_encoding_normalizer)
importlib.reload(step_2_boilerplate_remover)
importlib.reload(step_3_sentence_segmenter)
importlib.reload(step_4_citation_normalizer)
importlib.reload(step_5_embedding_generator)
importlib.reload(step_6_bertopic)
importlib.reload(step_7_statistics_generator)
importlib.reload(step_8_visualizer)
importlib.reload(step_9_xml_exporter)
importlib.reload(step_10_search_index)

from pipeline_step import PipelineStep
from pipeline_manager import PipelineManager
from step_0_pdf_reader import PdfReader, PdfReaderInput
from step_1_encoding_normalizer import EncodingNormalizer
from step_2_boilerplate_remover import BoilerplateRemover
from step_3_sentence_segmenter import SentenceSegmenter
from step_4_citation_normalizer import CitationNormalizer
from step_5_embedding_generator import EmbeddingGenerator
from step_6_bertopic import BerTopicStep
from step_7_statistics_generator import StatisticsGenerator
from step_8_visualizer import Visualizer
from step_9_xml_exporter import XmlExporter
from step_10_search_index import SumulaSearchIndex

# Suppress warnings and verbose loggers
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(name)s - %(levelname)s - %(message)s")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("umap").setLevel(logging.ERROR)
logging.getLogger("numba").setLevel(logging.ERROR)

Note: you may need to restart the kernel to use updated packages.


2026-03-17 17:13:46.363828: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Pipeline Configuration

All tunable parameters are defined here as a single source of truth. `random_seed` propagates to every step that requires stochastic reproducibility (UMAP, any future fine-tuned model).

In [2]:
# Global parameters
random_seed = 42  # PARAMETER: single source of randomness for all components

# Step 0
pdf_path = Path("./datasets/Súmulas - STJ.pdf")  # PARAMETER: path to the input PDF

ocr_fallback = True  # PARAMETER: attempt OCR for scanned pages
min_page_text = 50  # PARAMETER: minimum characters per page before OCR fallback

# Step 2
tfidf_threshold = 0.92  # PARAMETER: cosine similarity threshold for boilerplate detection

# Step 3
min_sentence_tokens = 5  # PARAMETER: minimum tokens for a sentence to be retained

# Step 5
embedding_batch_size = 16  # PARAMETER: sentences per inference batch
max_tokens = 512  # PARAMETER: BERT context window
overlap_tokens = 64  # PARAMETER: sliding window overlap
embedding_model_name = "neuralmind/bert-large-portuguese-cased"  # PARAMETER: HuggingFace model for sentence embeddings

# Step 6
bertopic_n_neighbors = 15  # PARAMETER: UMAP neighborhood size for BERTopic
bertopic_min_cluster = 2   # PARAMETER: HDBSCAN minimum cluster size - minimum valid value
bertopic_top_words = 5     # PARAMETER: number of keywords per topic label

# Checkpoint directory
checkpoint_dir = Path("./pipeline/checkpoints")

print(f"Random seed: {random_seed}")
print(f"Input PDF: {pdf_path}")
print(f"Checkpoint dir: {checkpoint_dir}")

Random seed: 42
Input PDF: datasets/Súmulas - STJ.pdf
Checkpoint dir: pipeline/checkpoints


### Build Pipeline

Instantiate all step processors and register them with the PipelineManager. The manager will execute each step in order, saving outputs to disk so that expensive steps can be skipped on subsequent runs.

In [3]:
steps = [
    PdfReader(min_text_length=min_page_text),
    EncodingNormalizer(),
    BoilerplateRemover(tfidf_threshold=tfidf_threshold),
    SentenceSegmenter(min_tokens=min_sentence_tokens),
    CitationNormalizer(),
    EmbeddingGenerator(
        max_tokens=max_tokens,
        overlap_tokens=overlap_tokens,
        batch_size=embedding_batch_size,
        model_name=embedding_model_name,
    ),
    BerTopicStep(
        n_neighbors=bertopic_n_neighbors,
        min_cluster_size=bertopic_min_cluster,
        top_n_words=bertopic_top_words,
        random_state=random_seed,
    ),
    StatisticsGenerator(),
]
manager = PipelineManager(checkpoint_dir=checkpoint_dir, steps=steps)
status = manager.checkpoint_status()
print("Checkpoint status:")
for step_num, exists in status.items():
    print(f"  Step {step_num:2d}: {'cached' if exists else 'pending'}")

Checkpoint status:
  Step  0: cached
  Step  1: cached
  Step  2: cached
  Step  3: cached
  Step  4: cached
  Step  5: pending
  Step  6: pending
  Step  7: pending


**Step 0. Read Text from PDF**

pdfplumber extracts the digital text layer directly from the PDF. When a page contains fewer than `min_page_text` characters (indicating a scanned image page), Tesseract OCR is invoked on a 300 DPI rasterisation. This dual approach handles mixed-mode documents common in Brazilian court archives.

In [4]:
pdf_input = PdfReaderInput(
    pdf_path=pdf_path,
    ocr_fallback=ocr_fallback,
    min_text_length=min_page_text,
)
step0_output = manager.run_step(0, pdf_input)
print(f"Pages: {step0_output.page_count}")
print(f"OCR pages: {step0_output.ocr_pages}")
print(f"Text length: {len(step0_output.raw_text):,} characters")
print(f"Preview: {step0_output.raw_text[:300]!r}")

pipeline.manager - INFO - Step 0 loaded from checkpoint


Pages: 2642
OCR pages: []
Text length: 5,391,932 characters
Preview: 'SUMSÁRIO úmulas Organizadas por\nRamo do Direito\nSUMÁRIO\nDIREITO ADMINISTRATIVO - ÁGUA E ESGOTO 30\nSúmula 407 30\nSúmula 412 33\nDIREITO ADMINISTRATIVO - ANISTIA POLÍTICA 35\nSúmula 624 35\nSúmula 647 38\nDIREITO ADMINISTRATIVO - ATIVIDADE FARMACÊUTICA 43\nSúmula 120 43\nSúmula 275 46\nSúmula 413 50\nSúmula 5'


**Step 1. Encoding Normalization**

ftfy repairs mojibake and other encoding artifacts produced by legacy PDF generators (common in Brazilian public sector documents). NFC normalization unifies code-point representations so that subsequent regex and tokenisation steps behave deterministically. All replacements are logged for audit purposes.

In [5]:
step1_output = manager.run_step(1, step0_output)
print(f"Replacements applied: {len(step1_output.replacements)}")
if step1_output.replacements:
    print("Sample replacements:")
    for orig, fixed in step1_output.replacements[:5]:
        print(f"  {orig!r} → {fixed!r}")
print(f"Clean text length: {len(step1_output.clean_text):,} characters")

pipeline.manager - INFO - Step 1 loaded from checkpoint


Replacements applied: 0
Clean text length: 5,391,932 characters


**Step 2. Boilerplate Removal**

Legal documents contain recurring non-informative headers, footers, and journal citation blocks. Three passes are applied: (1) regex patterns for deterministic boilerplate like page headers (`Súmulas Organizadas por Ramo do Direito`); (2) TF-IDF cosine similarity for near-duplicate paragraphs; (3) reference line stripping removes legislative citation blocks (`DJ DATA:`, `RSSTJ VOL.:`, `RSTJ VOL.:`, `scon.stj.jus.br/` lines) that would dilute embeddings. Boilerplate is replaced with `<BOILERPLATE>` to preserve positional context; reference lines are deleted entirely.

In [6]:
step2_output = manager.run_step(2, step1_output)
print(f"Segments replaced: {step2_output.removed_count}")
print(f"TF-IDF threshold used: {step2_output.tfidf_threshold}")
print(f"Filtered text length: {len(step2_output.filtered_text):,} characters")

pipeline.manager - INFO - Step 2 loaded from checkpoint


Segments replaced: 16270
TF-IDF threshold used: 0.92
Filtered text length: 4,950,655 characters


**Step 3. Súmula Segmentation**

Each súmula block — from its header line (`Súmula N`) to the next súmula header — is extracted as a single semantic unit. This preserves the full context of each legal rule (header + body text) as one embedding input, providing BERTimbau with richer context than individual sentences. Segments shorter than `min_sentence_tokens` words are discarded. When no súmula boundaries are detected, the step falls back to double-newline paragraph splitting.

In [7]:
step3_output = manager.run_step(3, step2_output)
print(f"Sentences extracted: {step3_output.sentence_count:,}")
print("\nSample sentences:")
for i, sent in enumerate(step3_output.sentences[:3]):
    print(f"  [{i+1}] {sent[:120]!r}")

pipeline.manager - INFO - Step 3 loaded from checkpoint


Sentences extracted: 288

Sample sentences:
  [1] 'SUMÁRIO\nDIREITO ADMINISTRATIVO - ÁGUA E ESGOTO 30\nSúmula 407 30\nSúmula 412 33\nDIREITO ADMINISTRATIVO - ANISTIA POLÍTICA '
  [2] 'SÚMULA 412\nDIREITO ADMINISTRATIVO - ÁGUA E ESGOTO\nEnunciado: Órgão Julgador:\nA ação de repetição de indébito de tarifas '
  [3] 'SÚMULA 647\nDIREITO ADMINISTRATIVO - ANISTIA POLÍTICA\nEnunciado: Órgão Julgador:\nSão imprescritíveis as ações indenizatór'


**Step 4. Citation Normalization**

Legal citations (article references, process numbers, case references, súmula numbers) are idiosyncratic strings that would fragment embedding space without normalization. Each citation type is replaced by a typed token (`<ART_REF>`, `<PROC_NUM>`, `<CASE_REF>`, `<SUMULA_REF>`), enabling BERT to focus on rhetorical context. Original citations are stored in `citation_metadata` for reconstruction.

In [8]:
step4_output = manager.run_step(4, step3_output)
total_citations = sum(len(m) for m in step4_output.citation_metadata)
print(f"Sentences processed: {len(step4_output.sentences):,}")
print(f"Total citations normalized: {total_citations:,}")
print("\nSample normalized sentences:")
for i, (sent, meta) in enumerate(zip(step4_output.sentences[:3], step4_output.citation_metadata[:3])):
    print(f"  [{i+1}] {sent[:120]!r}")
    if meta:
        print(f"       Citations: {meta}")

pipeline.manager - INFO - Step 4 loaded from checkpoint


Sentences processed: 288
Total citations normalized: 525

Sample normalized sentences:
  [1] 'SUMÁRIO\nDIREITO ADMINISTRATIVO - ÁGUA E ESGOTO 30\nSúmula 407 30\nSúmula 412 33\nDIREITO ADMINISTRATIVO - ANISTIA POLÍTICA '
       Citations: {'<ART_REF>': 'artigo 4º,\nda Lei 6.528/78 e artigos 11 caput, 11, § 2º e 32 do Decreto nº 82', '<CASE_REF>': 'REsp 485842'}
  [2] 'SÚMULA 412\nDIREITO ADMINISTRATIVO - ÁGUA E ESGOTO\nEnunciado: Órgão Julgador:\nA ação de repetição de indébito de tarifas '
       Citations: {'<ART_REF>': 'ART. 177 DO CÓDIGO CIVIL DE 1916 [...] Os serviços públicos', '<CASE_REF>': 'REsp 1467148'}
  [3] 'SÚMULA 647\nDIREITO ADMINISTRATIVO - ANISTIA POLÍTICA\nEnunciado: Órgão Julgador:\nSão imprescritíveis as ações indenizatór'
       Citations: {'<ART_REF>': 'ART. 2. DO DECRETO N. 20.377/31, SEGUNDO O QUAL O COMÉRCIO', '<CASE_REF>': 'REsp 35351'}


**Step 5. Embedding with BERTimbau Large** [STEP CAN BE SKIPPED - re-runs use checkpoint]

`neuralmind/bert-large-portuguese-cased` is the large variant of BERTimbau, trained on Portuguese Wikipedia and news corpora. Mean pooling over all non-padding token hidden states produces a single 1024-dim vector per sentence. The larger architecture captures richer linguistic representations than the base variant. Sentences exceeding 512 tokens are processed with a sliding window of `overlap_tokens` to prevent context loss at boundaries.

In [ ]:
step5_output = manager.run_step(5, step4_output)
print(f"Model: {step5_output.model_name}")
print(f"Embedding dimension: {step5_output.embedding_dim}")
print(f"Sentences embedded: {len(step5_output.embedded_sentences):,}")
sample_emb = step5_output.embedded_sentences[0].embedding
print(f"Sample vector norm: {float(np.linalg.norm(sample_emb)):.4f}")

pipeline.step_5 - INFO - Step 5 [Embedding Generator] started
httpx - INFO - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neuralmind/bert-large-portuguese-cased/aa302f6ea73b759f7df9cad58bd272127b67ec28/config.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/neuralmind/bert-large-portuguese-cased/aa302f6ea73b759f7df9cad58bd272127b67ec28/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

httpx - INFO - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neuralmind/bert-large-portuguese-cased/aa302f6ea73b759f7df9cad58bd272127b67ec28/tokenizer_config.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/neuralmind/bert-large-portuguese-cased/aa302f6ea73b759f7df9cad58bd272127b67ec28/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/neuralmind/bert-large-portuguese-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/neuralmind/bert-large-portuguese-cased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/main/vocab.txt "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neuralmind/bert-large-portuguese-cased/aa302f6ea73b759f7df9cad58bd272127b67ec28/vocab.txt "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/neuralmind/bert-large-portuguese-cased/aa302f6ea73b759f7df9cad58bd272127b67ec28/vocab.txt "HTTP/1.1 200 OK"


...

httpx - INFO - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neuralmind/bert-large-portuguese-cased/aa302f6ea73b759f7df9cad58bd272127b67ec28/config.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
httpx - INFO - HTTP Request: GET htt

pytorch_model.bin:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

httpx - INFO - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/neuralmind/bert-large-portuguese-cased "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/neuralmind/bert-large-portuguese-cased/commits/main "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/neuralmind/bert-large-portuguese-cased/discussions?p=0 "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/neuralmind/bert-large-portuguese-cased/commits/refs%2Fpr%2F6 "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/refs%2Fpr%2F6/model.safetensors.index.json "HTTP/1.1 404 Not Found"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/refs%2Fpr%2F6/model.safet

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

**Step 6. BERTopic Topic Modeling** [STEP CAN BE SKIPPED - re-runs use checkpoint]

BERTopic runs UMAP in 5D (cosine metric, fixed `random_state`) for HDBSCAN clustering, then fits a separate 2D UMAP for visualization only. HDBSCAN discovers density-based topic clusters. c-TF-IDF auto-generates keyword labels. Outlier sentences (noise) are reassigned to their nearest topic centroid via `reduce_outliers(strategy="embeddings")`, eliminating noise entirely. Topic labels follow the three-level format: `Branch > Subject > c-TF-IDF_subtype`, where Branch and Subject are extracted from the document's `DIREITO RAMO - SUBJECT` header via majority vote per cluster.

In [ ]:
step6_output = manager.run_step(6, step5_output)
print(f"Total topics found: {step6_output.total_topics}")
print(f"Total noise sentences: {step6_output.noise_count:,}")
print("\nTopic labels discovered:")
for topic_id, label in sorted(step6_output.topic_labels.items()):
    sentences_in_topic = sum(1 for s in step6_output.topiced_sentences if s.topic_id == topic_id)
    print(f"  Topic {topic_id:3d}: {label} ({sentences_in_topic} sentences)")

manager._steps[8] = Visualizer(
    bertopic_output=step6_output,
    output_dir=Path("./pipeline/figures"),
)
manager._steps[9] = XmlExporter(
    bertopic_output=step6_output,
    results_dir=Path("./pipeline/results"),
)

**Step 7. Generate Statistics**

For each non-noise BERTopic topic, the statistics step computes: sentence frequency, the representative sentence (highest cosine similarity to the centroid), non-representative sentence (lowest cosine similarity to centroid), and mean intra-topic cosine similarity. Topic labels come directly from BERTopic c-TF-IDF auto-generation — no BERTimbau re-inference or manual seed phrases are needed. Cross-topic centroid similarity reveals thematic overlap across the document.

In [ ]:
step7_output = manager.run_step(7, step6_output)
print(f"Total topics analysed: {step7_output.total_clusters}")
print("\nTopic frequency (auto-labeled):")
for topic_label, count in sorted(step7_output.topic_frequency.items(), key=lambda x: -x[1]):
    bar = "#" * (count // max(1, max(step7_output.topic_frequency.values()) // 20))
    print(f"  {topic_label:<30} {count:5d}  {bar}")
print("\nTop 5 topics by frequency:")
top5 = sorted(step7_output.cluster_stats, key=lambda s: s.frequency, reverse=True)[:5]
for stats in top5:
    print(f"  [topic-{stats.topic_id} / {stats.topic_label}] freq={stats.frequency} intra_sim={stats.intra_similarity:.3f}")
    print(f"    Representative:     {stats.representative_sentence[:100]!r}")
    print(f"    Non-representative: {stats.non_representative_sentence[:100]!r}")

**Step 8. Visualization**

Three complementary static plots are produced:
1. **UMAP scatter** — 2D UMAP coordinates from BERTopic coloured by topic using tab20 colormap, noise sentences in gray
2. **Frequency bar chart** — sentence count per topic sorted descending
3. **Topic label frequency chart** — total sentence count per auto-generated topic label

All plots are saved to `./pipeline/figures/` for inclusion in reports.

In [ ]:
step8_output = manager.run_step(8, step7_output)
for fig in step8_output.figures:
    plt.show()
print("\nFigures saved to:")
for path in step8_output.saved_paths:
    print(f"  {path}")

**Step 9. XML Export**

Each non-noise BERTopic topic is exported to a dedicated XML file under `pipeline/results/`, named after its sanitized auto-generated topic label. Sentences carry `topic_id` and `probability` attributes and are sorted by topic_id then probability descending.

In [ ]:
step9_output = manager.run_step(9, step7_output)
print(f"XML files exported to: {step9_output.results_dir}")
for topic_label, count in sorted(step9_output.sentence_counts.items(), key=lambda x: -x[1]):
    print(f"  {topic_label:<30} {count:5d} sentences")

### Frequency Table Summary

A structured frequency table consolidates all cluster statistics for downstream consumption (e.g., export to CSV, API response, or reporting dashboard).

In [ ]:
import pandas as pd

topic_clusters: dict[str, list] = {}
for s in step7_output.cluster_stats:
    topic_clusters.setdefault(s.topic_label, []).append(s)

rows = []
for topic_label, clusters in sorted(topic_clusters.items()):
    top_clusters = sorted(clusters, key=lambda s: -s.frequency)[:2]
    total_frequency = sum(s.frequency for s in clusters)
    for i, s in enumerate(top_clusters):
        rows.append({
            "topic_label": topic_label if i == 0 else "",
            "total_frequency": total_frequency if i == 0 else "",
            "sample_sentence": s.representative_sentence[:120],
        })

df_topics = pd.DataFrame(rows)
print("Topic frequency table:")
print(df_topics.to_string(index=False))

### Semantic Search

FAISS-based nearest-neighbour search over all Súmula embeddings. Enter any Portuguese legal text to retrieve the most semantically similar Súmulas.

In [ ]:
search_index = SumulaSearchIndex(
    embedding_output=step5_output,
    bertopic_output=step6_output,
)

query = "banco cobrou taxa de juros abusiva no contrato"  # PARAMETER: search query
results = search_index.search(query, top_k=10)

print(f"Query: {query!r}\n")
for r in results:
    print(f"  [{r.rank:2d}] sim={r.similarity:.4f}  topic={r.topic_label}")
    print(f"       {r.text[:150]!r}")
    print()

### Considerations

The following considerations apply before drawing conclusions from this pipeline:

**Step 5 — BERTimbau Large vs BERTimbau Base:**
`neuralmind/bert-large-portuguese-cased` provides richer 1024-dim embeddings compared to the 768-dim base variant. The additional capacity improves the separation of semantically distinct Súmulas in embedding space, which benefits both HDBSCAN cluster cohesion and FAISS search precision. A domain-specific legal Portuguese model would further improve results if one becomes publicly available.

**Dual-UMAP design:**
Clustering uses a 5D UMAP projection (cosine metric) to preserve richer semantic structure for HDBSCAN. A separate 2D UMAP is fitted on the original embeddings for visualization only. This separation avoids the quality degradation that occurs when a single 2D projection is used for both purposes.

**Cluster stability:**
HDBSCAN results are sensitive to the UMAP projection. Small corpora (fewer than ~200 sentences) may produce few or no clusters due to insufficient density. The pipeline is designed for corpora of hundreds to thousands of documents processed in batch.

**OCR quality:**
Tesseract accuracy on Brazilian court documents varies significantly with scan quality. Pre-processing steps (deskewing, denoising) upstream of this pipeline would improve OCR output and downstream text quality.

**Production readiness:**
This notebook is an exploratory MVP. A production deployment would separate the pipeline into microservices, add monitoring, and replace pickle checkpoints with a document database.

### Conclusions

#### Pipeline Summary

This MVP implements a complete 10-step NLP pipeline for semantic frequency analysis of Brazilian STJ Súmulas, from raw PDF ingestion through structured cluster statistics, visualizations, XML exports, and semantic search.

#### Key Technical Conclusions

- **Modular architecture**: Each step is independently testable and checkpointed, enabling iterative improvement without re-running expensive upstream operations
- **Reference line stripping**: Removing legislative citation blocks (DJ DATA, RSSTJ VOL, scon.stj.jus.br) from Step 2 produces cleaner embeddings with higher intra-cluster similarity
- **BERTimbau Large**: The 1024-dim large variant provides richer embeddings than the base 768-dim model, improving separation of semantically distinct Súmulas in embedding space
- **Dual-UMAP design**: Clustering in 5D preserves more semantic structure for HDBSCAN; 2D projection is fitted separately for visualization without compromising cluster quality
- **Three-level topic labels**: `Branch > Subject > c-TF-IDF_subtype` combines STJ's own document classification (ground truth) with statistically discriminating sub-keywords per cluster
- **Outlier elimination**: `reduce_outliers(strategy="embeddings")` assigns all noise sentences to their nearest topic centroid, ensuring 100% corpus coverage in analytics
- **FAISS semantic search**: L2-normalized inner-product search over BERTimbau Large embeddings enables sub-second retrieval of semantically similar Súmulas for any Portuguese legal query

#### Next Steps

- Evaluate a legal-domain fine-tuned Portuguese BERT (when publicly available) against BERTimbau Large on held-out Súmula similarity pairs
- Scale the pipeline to process corpora of 10,000+ PDFs across multiple STJ volumes
- Add LLM-based sub-cluster summarization using the representative sentence as context
- Export cluster embeddings to a vector database for production-grade semantic search